# Posterior Predictive Checks: modelos extendidos

**Etapa del workflow bayesiano: Model Expansion (Gelman et al. 2020)**

El análisis en `04_predictive_checks_base_models` mostró que:
- Poisson falla en dispersión **y** en proporción de ceros ($p_B \approx 0$).
- Binomial Negativa captura la dispersión pero subestima los ceros ($p_B = 0.093$).

El diagnóstico apunta a **inflación de ceros**: hay hembras que estructuralmente no atraen satélites, y ese proceso generador es distinto del proceso de conteo. La expansión natural son los modelos de mezcla:

- **ZIP** (*Zero-Inflated Poisson*): $\pi \cdot \delta_0 + (1-\pi) \cdot \text{Poisson}(\lambda)$. Aborda los ceros pero hereda la estructura de varianza de Poisson.
- **ZINB** (*Zero-Inflated Negative Binomial*): $\pi \cdot \delta_0 + (1-\pi) \cdot \text{NegBin}(\mu, \phi)$. Aborda simultáneamente ceros y sobredispersión.

Ambos modelos tienen dos componentes con predictor lineal sobre `W_scaled`: uno logístico que modela $P(\text{cero estructural} \mid W)$, y uno de conteo (Poisson o NegBin) que modela cuántos satélites dado que no es cero estructural.

> **Prerequisito:** ejecutar `03_bayesian_inference` y tener `outputs/pois_idata.nc` y `outputs/neg_idata.nc` disponibles.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import arviz as az
import cmdstanpy
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
np.random.seed(42)

## 1. Datos y modelos base

Cargamos los datos y los `InferenceData` de Poisson y Binomial Negativa generados en `03_bayesian_inference`. Los modelos base son necesarios para la comparación LOO-CV final con los cuatro modelos.

In [ ]:
df = pd.read_csv('../data/agresti_crab_satellites_dataset.csv')
mean_W = df['W'].mean()
std_W  = df['W'].std()
df['W_scaled'] = (df['W'] - mean_W) / std_W
y_obs = df['Sa'].values

stan_data = {
    'N': len(df),
    'x': df['W_scaled'].values.tolist(),
    'y': y_obs.tolist(),
}

# Cargar modelos base para comparación LOO final
idata_pois = az.from_netcdf('../outputs/pois_idata.nc')
idata_nb   = az.from_netcdf('../outputs/neg_idata.nc')
# plot_ppc requiere nombre consistente entre observed_data y posterior_predictive
idata_pois.posterior_predictive = idata_pois.posterior_predictive.rename({'y_rep': 'y'})
idata_nb.posterior_predictive   = idata_nb.posterior_predictive.rename({'y_rep': 'y'})

print(f"N observaciones: {len(df)}")
print(f"prop. ceros: {np.mean(y_obs == 0):.3f}")
print(f"índice de dispersión: {np.var(y_obs)/np.mean(y_obs):.3f}")

## 2. Ajuste de ZIP y ZINB

Los modelos ZI tienen cuatro parámetros: `alpha_count` y `beta_count` (intercepto y pendiente del componente de conteo, liga log), y `alpha_zero` y `beta_zero` (intercepto y pendiente del componente de ceros, liga logit). ZINB agrega `phi` (sobredispersión).

El prior `exponential(1)` sobre `phi` es apropiado: es positivo, tiene media 1, y concentra masa en valores moderados de sobredispersión. Si los `.nc` ya existen en `outputs/`, el bloque los carga directamente sin re-ajustar Stan.

In [ ]:
import os

if os.path.exists('../outputs/zip_idata.nc') and os.path.exists('../outputs/zinb_idata.nc'):
    print("Cargando .nc existentes...")
    idata_zip  = az.from_netcdf('../outputs/zip_idata.nc')
    idata_zinb = az.from_netcdf('../outputs/zinb_idata.nc')
else:
    print("Ajustando modelos...")
    zip_model  = cmdstanpy.CmdStanModel(stan_file='../models/zip_model.stan')
    zinb_model = cmdstanpy.CmdStanModel(stan_file='../models/zinb_model.stan')

    zip_fit = zip_model.sample(
        data=stan_data, chains=4, iter_warmup=1000, iter_sampling=1000,
        show_progress=True,
    )
    zinb_fit = zinb_model.sample(
        data=stan_data, chains=4, iter_warmup=1000, iter_sampling=1000,
        show_progress=True,
    )

    idata_zip = az.from_cmdstanpy(
        posterior=zip_fit,
        posterior_predictive='y_rep',
        log_likelihood='log_lik',
        observed_data={'y': y_obs},
    )
    idata_zinb = az.from_cmdstanpy(
        posterior=zinb_fit,
        posterior_predictive='y_rep',
        log_likelihood='log_lik',
        observed_data={'y': y_obs},
    )
    idata_zip.to_netcdf('../outputs/zip_idata.nc')
    idata_zinb.to_netcdf('../outputs/zinb_idata.nc')

# Renombrar para plot_ppc
idata_zip.posterior_predictive  = idata_zip.posterior_predictive.rename({'y_rep': 'y'})
idata_zinb.posterior_predictive = idata_zinb.posterior_predictive.rename({'y_rep': 'y'})

print("ZIP  grupos:", list(idata_zip.groups()))
print("ZINB grupos:", list(idata_zinb.groups()))

## 3. Diagnósticos MCMC

Verificamos convergencia con el resumen de ArviZ: $\hat{R} \approx 1.00$ indica que las cuatro cadenas muestrearon la misma distribución, y ESS > 400 garantiza estimaciones estables. Los modelos ZI tienen más parámetros que Poisson o NegBin — si la señal en los datos es débil para el componente de ceros, la convergencia puede ser más lenta. Complementamos con trace plots.

In [ ]:
print("=== ZIP ===")
display(az.summary(idata_zip,
                   var_names=['alpha_count','beta_count','alpha_zero','beta_zero'],
                   round_to=3))

print("\n=== ZINB ===")
display(az.summary(idata_zinb,
                   var_names=['alpha_count','beta_count','phi','alpha_sm','alpha_zero','beta_zero'],
                   round_to=3))

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(12, 10))
az.plot_trace(idata_zip,
              var_names=['alpha_count','beta_count','alpha_zero','beta_zero'],
              axes=axes, compact=False)
fig.suptitle('Trace plots — ZIP', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(5, 2, figsize=(12, 12))
az.plot_trace(idata_zinb,
              var_names=['alpha_count','beta_count','phi','alpha_zero','beta_zero'],
              axes=axes, compact=False)
fig.suptitle('Trace plots — ZINB', fontsize=13)
plt.tight_layout()
plt.show()

## 4. PPCs visuales — distribución marginal

Comparamos los cuatro modelos lado a lado. Si ZIP y ZINB modelan correctamente la inflación de ceros, su nube de $y^{\text{rep}}$ debe ajustarse mejor al pico en cero y a la cola derecha observados en los datos.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

for ax, (idata, title) in zip(axes.flat, [
    (idata_pois, 'Poisson'),
    (idata_nb,   'Binomial Negativa'),
    (idata_zip,  'ZIP'),
    (idata_zinb, 'ZINB'),
]):
    az.plot_ppc(idata, ax=ax, num_pp_samples=100)
    ax.set_title(title)
    ax.set_xlabel('Número de satélites')
    ax.set_xlim(-0.5, 17)

plt.suptitle('PPC: Distribución marginal — cuatro modelos', fontsize=13)
plt.tight_layout()
plt.show()

## 5. PPCs con estadísticos de prueba

Aplicamos los mismos cuatro estadísticos del notebook `04_predictive_checks_base_models` a los cuatro modelos. La tabla y los histogramas permiten ver de forma sistemática qué dimensiones resuelve cada extensión — y cuáles siguen siendo problemáticas.

In [ ]:
def dispersion_index(y): return np.var(y) / np.mean(y)
def prop_zeros(y): return np.mean(y == 0)

test_stats = {
    'std':               np.std,
    'Índice dispersión': dispersion_index,
    'max':               np.max,
    'prop. ceros':       prop_zeros,
}

models = [
    ('Poisson', idata_pois),
    ('NegBin',  idata_nb),
    ('ZIP',     idata_zip),
    ('ZINB',    idata_zinb),
]

y_rep_dict = {}
for name, idata in models:
    arr = idata.posterior_predictive['y'].values
    y_rep_dict[name] = arr.reshape(-1, len(y_obs))

# Tabla comparativa
rows = []
for stat_name, stat_fn in test_stats.items():
    t_obs = stat_fn(y_obs)
    for name, y_rep_flat in y_rep_dict.items():
        t_rep = np.array([stat_fn(y_rep_flat[s]) for s in range(y_rep_flat.shape[0])])
        rows.append({
            'Modelo': name,
            'Estadístico': stat_name,
            'T(y_obs)': round(t_obs, 3),
            'E[T(y_rep)]': round(float(np.mean(t_rep)), 3),
            'Bayesian p-value': round(float(np.mean(t_rep >= t_obs)), 3),
        })

summary_df = pd.DataFrame(rows).sort_values(['Estadístico', 'Modelo'])
display(summary_df)

In [ ]:
fig, axes = plt.subplots(len(test_stats), len(models), figsize=(16, 4 * len(test_stats)))

for row, (stat_name, stat_fn) in enumerate(test_stats.items()):
    t_obs = stat_fn(y_obs)
    for col, (model_name, y_rep_flat) in enumerate(
        (n, y_rep_dict[n]) for n, _ in models
    ):
        t_rep = np.array([stat_fn(y_rep_flat[s]) for s in range(y_rep_flat.shape[0])])
        p_val = np.mean(t_rep >= t_obs)

        ax = axes[row, col]
        ax.hist(t_rep, bins=40, color='steelblue', alpha=0.7, label='T(y_rep)')
        ax.axvline(t_obs, color='tomato', lw=2, label=f'T(y_obs)={t_obs:.2f}')
        ax.set_title(f'{model_name} — {stat_name}\np = {p_val:.3f}', fontsize=9)
        ax.legend(fontsize=7)

plt.suptitle('PPCs con estadísticos de prueba — cuatro modelos', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 6. Comparación global — LOO-CV

La comparación final incluye los cuatro modelos. El ELPD ordena los modelos por capacidad predictiva out-of-sample; la diferencia ± SE indica si el ranking es estadísticamente claro o si los modelos son indistinguibles desde el punto de vista predictivo.

In [ ]:
comparison = az.compare({
    'Poisson': idata_pois,
    'NegBin':  idata_nb,
    'ZIP':     idata_zip,
    'ZINB':    idata_zinb,
})
display(comparison)

ax = az.plot_compare(comparison, figsize=(9, 3))
ax.set_title('Comparación LOO-CV — cuatro modelos')
plt.tight_layout()
plt.show()

## 7. Posterior del componente de ceros

El componente logístico modela $P(\text{cero estructural} \mid W)$: la probabilidad de que una hembra no atraiga satélites por razones estructurales. La hipótesis es que esta probabilidad disminuye con el tamaño — hembras más grandes atraen más, y es menos probable que no atraigan ninguno.

Las curvas semitransparentes son 200 muestras de la posterior — su amplitud refleja la incertidumbre sobre la pendiente logística. La curva de la media posterior muestra la tendencia central.

In [ ]:
W_range = np.linspace(df['W'].min(), df['W'].max(), 100)
W_scaled_range = (W_range - mean_W) / std_W

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, (name, idata) in zip(axes, [('ZIP', idata_zip), ('ZINB', idata_zinb)]):
    a0 = idata.posterior['alpha_zero'].values.flatten()
    b0 = idata.posterior['beta_zero'].values.flatten()

    # Curvas de theta para 200 muestras del posterior
    idx = np.random.choice(len(a0), 200, replace=False)
    for i in idx:
        theta = 1 / (1 + np.exp(-(a0[i] + b0[i] * W_scaled_range)))
        ax.plot(W_range, theta, color='steelblue', alpha=0.05)

    # Curva de la media posterior
    theta_mean = 1 / (1 + np.exp(-(a0.mean() + b0.mean() * W_scaled_range)))
    ax.plot(W_range, theta_mean, color='tomato', lw=2, label='Media posterior')

    ax.set_xlabel('Ancho de caparazón W (cm)')
    ax.set_ylabel(r'$P(\text{cero estructural} \mid W)$')
    ax.set_title(f'{name} — Componente de ceros')
    ax.legend()
    ax.set_ylim(0, 1)

plt.suptitle('Probabilidad de cero estructural vs ancho de caparazón', fontsize=12)
plt.tight_layout()
plt.show()

## 8. Conclusiones

### Resultados de PPCs

| Estadístico | $y^{\text{obs}}$ | Poisson | NegBin | ZIP | ZINB |
|---|---|---|---|---|---|
| std | 3.14 | $p_B = 0.000$ ❌ | $p_B = 0.892$ ✅ | $p_B = 0.011$ ⚠️ | $p_B = 0.635$ ✅ |
| Índice dispersión | 3.38 | $p_B = 0.000$ ❌ | $p_B = 0.951$ ✅ | $p_B = 0.008$ ❌ | $p_B = 0.691$ ✅ |
| max | 15 | $p_B = 0.072$ ⚠️ | $p_B = 0.963$ ✅ | $p_B = 0.014$ ⚠️ | $p_B = 0.624$ ✅ |
| **prop. ceros** | **0.358** | **$p_B = 0.000$ ❌** | **$p_B = 0.092$ ⚠️** | **$p_B = 0.514$ ✅** | **$p_B = 0.532$ ✅** |

### Hallazgos principales

**ZIP**: Resuelve la inflación de ceros ($p_B = 0.514$) pero hereda la estructura de varianza de Poisson — falla en std ($p_B = 0.011$) y en el índice de dispersión ($p_B = 0.008$). En datos con ambos problemas simultáneos, ZIP es una mejora parcial insuficiente.

**ZINB**: Captura simultáneamente inflación de ceros ($p_B = 0.532$) y sobredispersión ($p_B = 0.691$). Ningún estadístico de prueba lo rechaza — es el modelo más completo de los cuatro evaluados.

**Componente de ceros**: En ambos modelos ZI, `beta_zero < 0` — hembras con mayor ancho de caparazón tienen menor probabilidad de ser "cero estructural". Esto es interpretable y coherente con la biología: hembras más grandes atraen más satélites.

**LOO-CV**: El ranking esperado es ZINB > ZIP > NegBin > Poisson. La diferencia ZINB–NegBin cuantifica el valor de modelar explícitamente el proceso generador de ceros sobre y por encima de capturar la sobredispersión.

### Limitación persistente

El predictor es únicamente el ancho de caparazón (`W`). Variables como condición de la espina (`S`) y color (`C`) no fueron incluidas. Una extensión natural es un ZINB con múltiples predictores en ambos componentes — o un modelo jerárquico si hubiera agrupación natural en los datos.

---

## Extensiones y conexiones

### Quasi-Poisson: la alternativa frecuentista

El enfoque bayesiano que recorrimos no es el único camino para manejar la
sobredispersión en datos de conteo. Desde el paradigma frecuentista, el modelo
**quasi-Poisson** (Wedderburn 1974) extiende la regresión Poisson relajando el
supuesto Var[Y] = E[Y] para permitir:

$$\text{Var}[Y_i] = \phi \cdot E[Y_i], \quad \phi > 1$$

donde $\phi$ es un parámetro de dispersión estimado de los datos. El modelo trabaja
con una **función de cuasi-verosimilitud** (no requiere especificar una distribución
completa para Y).

En R: `glm(y ~ x, family = quasipoisson)`. En Python, no existe una implementación
nativa directa en `statsmodels`, lo que es en sí un argumento a favor de los modelos
con distribución explícita (NegBin, ZINB).

El análogo para datos de proporciones o tasas de conversión es el modelo
**quasi-binomial** — misma idea, función de varianza $\phi \cdot p(1-p)$.

La limitación de los modelos quasi: no producen una distribución predictiva completa,
lo que dificulta calcular intervalos de predicción calibrados o comparar modelos vía
LOO-CV. El marco bayesiano con NegBin o ZINB provee todo esto de forma natural.